# Labo 41 · Carnet Colab — Un agent ReAct sur GPU

Un agent ReAct autour d'un modèle ouvert (Qwen2.5-1.5B-Instruct), avec trois outils réels, une limite d'étapes, un journal, puis une évaluation sur une série de tâches.

**Avant de commencer** : *Exécution → Modifier le type d'exécution → GPU (T4)*, puis *Fichier → Enregistrer une copie dans Drive*.

À la fin, recopie tes résultats dans la case « Carnet Colab » de la page du labo.

In [ ]:
import subprocess, sys
subprocess.run([sys.executable, "-m", "pip", "install", "-q", "transformers", "accelerate"], check=False)
import re, json
import numpy as np
import torch
from transformers import AutoTokenizer, AutoModelForCausalLM
from sklearn.feature_extraction.text import TfidfVectorizer

nom = "Qwen/Qwen2.5-1.5B-Instruct"
decoupeur = AutoTokenizer.from_pretrained(nom)
modele = AutoModelForCausalLM.from_pretrained(nom, torch_dtype=torch.float16 if torch.cuda.is_available() else torch.float32)
modele = modele.to("cuda" if torch.cuda.is_available() else "cpu")

def generer(messages, temperature=0.0, max_jetons=200):
    texte = decoupeur.apply_chat_template(messages, tokenize=False, add_generation_prompt=True)
    entree = decoupeur(texte, return_tensors="pt").to(modele.device)
    options = dict(max_new_tokens=max_jetons, do_sample=temperature > 0, stop_strings=["Observation"], tokenizer=decoupeur)
    if temperature > 0:
        options["temperature"] = temperature
    sortie = modele.generate(**entree, **options)
    return decoupeur.decode(sortie[0][entree["input_ids"].shape[1]:], skip_special_tokens=True).split("Observation")[0].strip()

## 1. Les outils

Trois outils, chacun une fonction d'un seul argument texte : une calculatrice (sûre : seulement des chiffres et des opérateurs), l'horaire des cours et une recherche dans le règlement (le RAG du cours 38).

In [ ]:
HORAIRE = {"420-SN1-RE": "lundi et mercredi, 13 h à 16 h, local D-214", "201-NYA-05": "mardi, 8 h à 11 h, local B-102",
           "109-101-MQ": "jeudi, 10 h à 12 h, gymnase"}
REGLEMENT = [
    "Article 2. Absences. Au-delà de 10 % d'absences non motivées dans un cours, l'enseignant peut refuser l'accès à l'évaluation finale de ce cours.",
    "Article 3. Absence à une évaluation. L'étudiant absent à une évaluation doit fournir une pièce justificative dans les 5 jours ouvrables pour obtenir une reprise.",
    "Article 4. Retards dans les travaux. Un travail remis en retard perd 10 % de sa valeur par jour, jusqu'à un maximum de 5 jours, après quoi il reçoit la note zéro.",
    "Article 8. Note de passage. La note de passage d'un cours est 60 %.",
]
tfidf = TfidfVectorizer().fit(REGLEMENT)

def calculer(expression):
    if not re.fullmatch(r"[0-9+\-*/(). ]+", expression):
        raise ValueError("expression non permise : seulement des nombres et + - * / ( )")
    return round(eval(expression, {"__builtins__": {}}), 4)

def horaire(cours):
    if cours.strip() not in HORAIRE:
        raise ValueError(f"cours inconnu : {cours}. Cours connus : {', '.join(HORAIRE)}")
    return HORAIRE[cours.strip()]

def reglement(question):
    scores = (tfidf.transform([question]) @ tfidf.transform(REGLEMENT).T).toarray()[0]
    return REGLEMENT[int(np.argmax(scores))]

OUTILS = {"calculer": calculer, "horaire": horaire, "reglement": reglement}
print(calculer("100 * 0.9 ** 3"), horaire("201-NYA-05"), reglement("travail en retard"), sep="\n")

## 2. La boucle ReAct

**À toi** : complète `agent(question)`. Le message système (fourni) décrit le format attendu. À chaque étape : générer la suite, la lire avec `analyser` (exercice 41.1), exécuter l'outil (avec les erreurs comme observations), ajouter l'observation, et s'arrêter à la réponse finale ou après `MAX_ETAPES`. Garde un `journal` de chaque étape.

In [ ]:
SYSTEME = """Tu es l'assistant du Cégep Boréal. Tu résous la tâche par étapes, avec ces outils :
- calculer[expression] : calcule une expression arithmétique, par exemple calculer[100 * 0.9 ** 3]
- horaire[code du cours] : l'horaire d'un cours, par exemple horaire[420-SN1-RE]
- reglement[question] : l'article du règlement pédagogique le plus pertinent

À chaque étape, écris exactement :
Pensée : ce que tu sais et ce qu'il te manque
Action : outil[argument]
Puis attends l'observation. Quand tu as la réponse, écris :
Réponse finale : la réponse, en une phrase."""
MAX_ETAPES = 6

def analyser(sortie):
    for ligne in sortie.split("\n"):
        m = re.match(r"Réponse finale\s*:\s*(.*)$", ligne.strip())
        if m:
            return ("finale", m.group(1).strip())
        m = re.match(r"Action\s*:\s*(\w+)\[(.*)\]\s*$", ligne.strip())
        if m:
            return ("action", m.group(1), m.group(2).strip())
    return ("invalide",)

def agent(question, temperature=0.0, afficher=True):
    # À toi : retourne (reponse ou None, nombre d'étapes, journal)
    pass

## 3. Évaluer l'agent

**À toi** : lance l'agent 3 fois (T = 0.7) sur chacune des 6 tâches. Une tâche est réussie si la réponse finale contient la valeur attendue. Calcule le taux de réussite, le nombre moyen d'étapes des réussites, et la fiabilité (la proportion des tâches réussies aux **3** essais). Examine un échec : mauvais outil, mauvais argument, format invalide ou hallucination?

In [ ]:
taches = [
    ("À quelle heure commence le cours 201-NYA-05?", ["8 h", "8h"]),
    ("Quelle est la note de passage d'un cours?", ["60"]),
    ("Combien de jours ai-je pour justifier une absence à un examen?", ["5"]),
    ("Combien font 17.5 % de 2340 $?", ["409.5"]),
    ("Un travail de 80 points est remis avec 2 jours de retard. Combien de points vaut-il au maximum?", ["64"]),
    ("Dans quel local a lieu le cours 420-SN1-RE?", ["D-214"]),
]
# À toi

## À remettre

Recopie dans la case « Carnet Colab » de la page du labo : la trace complète d'une tâche réussie et d'une tâche ratée, le taux de réussite, le nombre moyen d'étapes, le nombre de tâches fiables, et ton explication de l'échec. Pour finir : à quelles tâches du cégep confierais-tu un tel agent, et lesquelles garderais-tu pour un humain?